# Invoking a Model with InvokeModel

Now that we can list models, let's actually call one. This uses the `bedrock-runtime` client (the data plane that runs inference) and Anthropic's Claude Sonnet 4.5.

Steps:
1. Build the request payload (Anthropic Messages format)
2. Call `invoke_model`
3. Parse the response and read the generated text
4. Look at the full response structure

## 1. Build the payload

Each model family has its own request format. Claude uses the Anthropic Messages format: an `anthropic_version`, a `max_tokens` limit, and a list of `messages`.

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 5000,
    "messages": [
        {
            "role": "user",
            "content": "Create a script to resize images"
        }
    ]
})

## 2. Invoke the model

The `modelId` here is an inference profile (the `global.` prefix routes the request across regions). `accept` and `contentType` tell Bedrock we're sending and receiving JSON.

In [ ]:
response = bedrock_runtime.invoke_model(
    body=body,
    modelId="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    accept="application/json",
    contentType="application/json"
)

## 3. Read the generated text

The response body is a stream, so we read it and parse the JSON. The generated text lives at `content[0]["text"]`.

In [ ]:
response_body = json.loads(response.get("body").read())
print(response_body["content"][0]["text"])

## 4. The response structure

Beyond the generated text, the response includes metadata: the model, why it stopped, and token usage. Below we show the full structure with the long generated text replaced by a placeholder (matches the slide).

In [ ]:
import copy

view = copy.deepcopy(response_body)
view["content"][0]["text"] = "<model-output>"  # placeholder for brevity

print(json.dumps(view, indent=4))